# trajectory-HRS

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings('ignore')

# =========================
# Step 0: Paths + Params
# =========================
INPUT_CSV = r"./input/HRS1.csv"
OUT_DIR   = r"./output/HRS"
os.makedirs(OUT_DIR, exist_ok=True)

FIXED_K = 3  # Force 3 trajectories

OUT_DENDRO     = os.path.join(OUT_DIR, "dendrogram_HRS_Hierarchical_preDementia_byAge.pdf")
OUT_WITH_GROUP_age = os.path.join(OUT_DIR, "output_with_group_HRS_Hierarchical_preDementia_byAge.csv")
OUT_DIFF       = os.path.join(OUT_DIR, "score_diff_HRS_Hierarchical_preDementia_byAge.csv")

# =========================
# Step 1: Load data
# =========================
df = pd.read_csv(INPUT_CSV)

# Basic type safety
for col in ['score', 'wave', 'age', 'dementia']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Ensure required columns
required_cols = ['ID', 'wave', 'age', 'score']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}")

# Drop rows missing key variables
df = df.dropna(subset=['ID', 'wave', 'age', 'score']).copy()

# =========================
# Step 1b: Handle duplicates (ID-wave)
# =========================
# score: mean
# dementia: max (if any record at that wave indicates dementia -> 1)
# age: mean
agg_map = {'score': 'mean', 'age': 'mean'}
if 'dementia' in df.columns:
    agg_map['dementia'] = 'max'
else:
    df['dementia'] = 0
    agg_map['dementia'] = 'max'

df = (
    df.groupby(['ID', 'wave'], as_index=False)
      .agg(agg_map)
)

# Ensure dementia is 0/1-like
df['dementia'] = df['dementia'].fillna(0)

# =========================
# Step 1c: Dynamic truncation BEFORE first dementia (by AGE)
# =========================
first_event = (
    df.loc[df['dementia'] == 1]
      .sort_values(['ID', 'wave'])
      .groupby('ID', as_index=False)
      .first()[['ID', 'wave', 'age']]
      .rename(columns={'wave': 'first_dem_wave', 'age': 'first_dem_age'})
)

df = df.merge(first_event, on='ID', how='left')

# Keep:
# - never dementia -> keep all
# - dementia -> keep only observations strictly BEFORE first dementia age
df = df[(df['first_dem_age'].isna()) | (df['age'] < df['first_dem_age'])].copy()

# Drop helper columns
df.drop(columns=['first_dem_wave', 'first_dem_age'], inplace=True)

# =========================
# Step 1d: Keep IDs with >=2 measurements AFTER truncation
# =========================
measurement_counts = df.groupby('ID').size()
ids_with_at_least_2 = measurement_counts[measurement_counts >= 2].index
df = df[df['ID'].isin(ids_with_at_least_2)].copy()
df = df.sort_values(['ID', 'wave']).copy()

# =========================
# Step 2: Interpolate to common wave grid (NO extrapolated trend)
# =========================
min_time, max_time = df['wave'].min(), df['wave'].max()
time_grid = np.linspace(min_time, max_time, num=20)

def interpolate_trajectory(group: pd.DataFrame) -> np.ndarray:
    """
    Linear interpolation within observed waves.
    Outside observed range: endpoint carry (NO extrapolated trend).
    """
    waves = group['wave'].values
    scores = group['score'].values

    if len(group) < 2:
        return np.full(len(time_grid), scores[0])

    f = interp1d(waves, scores, kind='linear', bounds_error=False)
    y = f(time_grid)

    # endpoint carry
    y[time_grid < waves.min()] = scores[np.argmin(waves)]
    y[time_grid > waves.max()] = scores[np.argmax(waves)]
    return y

unique_ids = df['ID'].unique()
smoothed_trajectories = []
derivative_features = []

for id_val in unique_ids:
    group = df[df['ID'] == id_val]
    interp_scores = interpolate_trajectory(group)

    # Savitzky-Golay smoothing
    if len(interp_scores) >= 5:
        smoothed = savgol_filter(interp_scores, window_length=5, polyorder=2)
    else:
        smoothed = interp_scores

    # Derivative (slope)
    deriv = np.gradient(smoothed, time_grid)

    smoothed_trajectories.append(smoothed)
    derivative_features.append(deriv)

X_smoothed = np.array(smoothed_trajectories)
X_deriv = np.array(derivative_features)
X = np.hstack([X_smoothed, X_deriv])

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================
# Step 3: Hierarchical clustering + dendrogram
# =========================
linkage_matrix = linkage(X_scaled, method='ward', metric='euclidean')

plt.figure(figsize=(10, 7))
dendrogram(linkage_matrix)
plt.title('Dendrogram (Pre-dementia only, Smoothed + Derivatives)')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.savefig(OUT_DENDRO, format='pdf', bbox_inches='tight')
plt.close()

# =========================
# Step 3b: Force fixed K
# =========================
hierarchical = AgglomerativeClustering(n_clusters=FIXED_K, linkage='ward')
cluster_labels = hierarchical.fit_predict(X_scaled)

cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
print("Cluster sizes:\n", cluster_sizes)

# (Optional) print the silhouette as a reference (not used for selecting K)
try:
    sil = silhouette_score(X_scaled, cluster_labels)
    print(f"Silhouette (fixed k={FIXED_K}): {sil:.3f}")
except Exception as e:
    print("Silhouette calculation failed:", e)

# =========================
# Step 4: Assign groups back and export
# =========================
id_to_group = dict(zip(unique_ids, cluster_labels))
df['group'] = df['ID'].map(id_to_group)

df.to_csv(OUT_WITH_GROUP_age, index=False)
print(f"Saved: {OUT_WITH_GROUP_age}")

# =========================
# Additional: score difference for IDs with >=3 measurements AFTER truncation
# =========================
measurement_counts2 = df.groupby('ID').size()
ids_with_at_least_3 = measurement_counts2[measurement_counts2 >= 3].index

diff_data = []
for id_val in ids_with_at_least_3:
    id_data = df[df['ID'] == id_val].sort_values('wave')
    score1 = id_data['score'].iloc[0]
    score2 = id_data['score'].iloc[1]
    diff_data.append({'ID': id_val, 'score_D': score2 - score1, 'score1': score1})

diff_df = pd.DataFrame(diff_data)
diff_df.to_csv(OUT_DIFF, index=False)
print(f"Saved: {OUT_DIFF}")


In [ ]:
# ============================================================
# Step 0: 参数设置
# ============================================================

FIXED_K = 3 
GRID_NUM = 20

OUT_WITH_GROUP = os.path.join(OUT_DIR, "output_with_group_HRS_Hierarchical.csv")
OUT_SURVIVAL   = os.path.join(OUT_DIR, "survival_data_wide_HRS_Hierarchical.csv")

OUT_FIG1 = os.path.join(OUT_DIR, "mean_trajectories_HRS_Hierarchical_preDementia_byAge.pdf")
OUT_FIG2 = os.path.join(OUT_DIR, "smooth_fitted_trajectories_HRS_Hierarchical_preDementia_byAge.pdf")
OUT_FIG3 = os.path.join(OUT_DIR, "individual_trajectories_HRS_Hierarchical_preDementia_byAge.pdf")

# ============================================================
# Step 1: Loading and cleaning
# ============================================================
df_raw = pd.read_csv(INPUT_CSV)
for col in ['score', 'wave', 'age', 'dementia']:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df = df_raw.dropna(subset=['ID', 'wave', 'age', 'score']).copy()

# ============================================================
# Step 1c: Dynamic truncation (only trajectories before diagnosis are retained)
# ============================================================
first_event = (
    df.loc[df['dementia'] == 1]
      .sort_values(['ID', 'age'])
      .groupby('ID', as_index=False)
      .first()[['ID', 'age']]
      .rename(columns={'age': 'first_dem_age'})
)
df = df.merge(first_event, on='ID', how='left')
df = df[(df['first_dem_age'].isna()) | (df['age'] < df['first_dem_age'])].copy()

# Screening people with at least 2 measurements (changeable 3)
measurement_counts = df.groupby('ID').size()
ids_with_min_obs = measurement_counts[measurement_counts >= 2].index
df = df[df['ID'].isin(ids_with_min_obs)].copy()

# ============================================================
# Step 2: Interpolation and smoothing logic (fully reserved)
# ============================================================
min_time, max_time = df['wave'].min(), df['wave'].max()
time_grid = np.linspace(min_time, max_time, num=GRID_NUM)

def interpolate_trajectory(group: pd.DataFrame) -> np.ndarray:
    """
    Linear interpolation. Out of view: endpoint carry (NO extrapolated trend).
    """
    waves = group['wave'].values
    scores = group['score'].values
    if len(group) < 2:
        return np.full(len(time_grid), scores[0])
    
    f = interp1d(waves, scores, kind='linear', bounds_error=False)
    y = f(time_grid)
    # Endpoint carry (your original logic)
    y[time_grid < waves.min()] = scores[np.argmin(waves)]
    y[time_grid > waves.max()] = scores[np.argmax(waves)]
    return y

unique_ids = df['ID'].unique()
smoothed_trajectories = []
derivative_features = []

for id_val in unique_ids:
    group = df[df['ID'] == id_val].sort_values('wave')
    interp_scores = interpolate_trajectory(group)

    # Savitzky-Golay smoothing
    if len(interp_scores) >= 5:
        smoothed = savgol_filter(interp_scores, window_length=5, polyorder=2)
    else:
        smoothed = interp_scores

    # Extract the derivative (slope)
    deriv = np.gradient(smoothed, time_grid)
    smoothed_trajectories.append(smoothed)
    derivative_features.append(deriv)

# Feature stacking logic
X_smoothed = np.array(smoothed_trajectories)
X_deriv = np.array(derivative_features)
X = np.hstack([X_smoothed, X_deriv])

# standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================
# Step 4: Clustering (fix parameter name: affinity -&gt; metric)
# ============================================================
hierarchical = AgglomerativeClustering(n_clusters=FIXED_K, linkage='ward', metric='euclidean')
cluster_labels = hierarchical.fit_predict(X_scaled)

id_to_group = dict(zip(unique_ids, cluster_labels))
df['group'] = df['ID'].map(id_to_group)
df.to_csv(OUT_WITH_GROUP, index=False)

# ============================================================
# Step 5: Landmark Analysis
# ============================================================
final_status = df_raw.sort_values(['ID', 'age']).groupby('ID').agg({
    'dementia': 'max',
    'age': 'max'
}).rename(columns={'age': 'final_obs_age', 'dementia': 'event_outcome'})

baseline_info = df.groupby('ID').agg({
    'age': 'max',
    'group': 'first'
}).rename(columns={'age': 'landmark_age_t0'})

survival_df = baseline_info.merge(final_status, on='ID', how='inner')
survival_df['survival_time'] = survival_df['final_obs_age'] - survival_df['landmark_age_t0']
survival_df = survival_df[survival_df['survival_time'] >= 0].copy()

survival_df.to_csv(OUT_SURVIVAL, index=True)
print(f"Survival analysis file saved: {OUT_SURVIVAL}")

# ============================================================
# Step 6: Drawing 
# ============================================================
sns.set_context("paper", font_scale=1.2) 
sns.set_style("whitegrid")
colors = sns.color_palette("tab10", n_colors=len(df['group'].unique()))

# --- FIG 1: Mean Trajectories with CI ---
plt.figure(figsize=(12, 8))
for i, g in enumerate(sorted(df['group'].dropna().unique())):
    gd = df[df['group'] == g]
    sns.lineplot(
        data=gd, x='age', y='score',
        color=colors[i], ci=95, linewidth=3,
        label=f'Group {g} (Mean with 95% CI)'
    )
plt.xlabel('Age (Years)', fontsize=14)
plt.ylabel('BrainVital8 Score', fontsize=14)
plt.title('Mean Trajectories by Group (Pre-dementia Only)', fontsize=16)
plt.legend(frameon=True)
plt.savefig(OUT_FIG1, format='pdf', bbox_inches='tight', dpi=300)
plt.close()

# --- FIG 2: Smooth Fitted Trajectories (Cubic Fit) ---
plt.figure(figsize=(12, 8))
for i, g in enumerate(sorted(df['group'].dropna().unique())):
    gd = df[df['group'] == g]
    sns.regplot(
        data=gd, x='age', y='score',
        order=3, ci=95, scatter=False,
        line_kws={'linewidth': 3}, color=colors[i],
        label=f'Group {g} (Cubic Fit with 95% CI)'
    )
plt.xlabel('Age (Years)', fontsize=14)
plt.ylabel('BrainVital8 Score', fontsize=14)
plt.title('Smooth Fitted Trajectories by Group (Pre-dementia Only)', fontsize=16)
plt.legend(frameon=True)
plt.savefig(OUT_FIG2, format='pdf', bbox_inches='tight', dpi=300)
plt.close()

# --- FIG 3: Individual Trajectories (Spaghetti Plot) ---
plt.figure(figsize=(12, 8))
alpha_value = 0.05

for i, g in enumerate(sorted(df['group'].dropna().unique())):
    gd = df[df['group'] == g]
    c = colors[i]

    # Background individual trajectories were drawn
    for id_val in gd['ID'].unique():
        id_data = gd[gd['ID'] == id_val]
        plt.plot(id_data['age'], id_data['score'], color=c, alpha=alpha_value, linewidth=0.8)

    # Plot foreground mean trajectories
    sns.lineplot(
        data=gd, x='age', y='score', 
        color=c, ci=None, linewidth=4,
        label=f'Group {g} Population Mean'
    )

plt.xlabel('Age (Years)', fontsize=14)
plt.ylabel('BrainVital8 Score', fontsize=14)
plt.title('Individual and Mean Trajectories by Group', fontsize=16)
plt.legend(frameon=True)
plt.savefig(OUT_FIG3, format='pdf', bbox_inches='tight', dpi=300)
plt.close()

print(f"All high quality PDF images have been saved to the directory:{OUT_DIR}")